# Real-Time Facial Emotion Detection with Deep Learning, CV & NLP 🚀

This Jupyter Notebook contains the COMPLETE end-to-end pipeline. 
Since you are running this in VS Code, you can execute these blocks one by one to train the model, test the NLP, and finally run the live webcam feed!

In [ ]:
# Run this cell to install dependencies if you haven't already!
!pip install tensorflow keras opencv-python numpy pandas matplotlib seaborn scikit-learn nltk ipywidgets pillow

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.tokenize import word_tokenize
import random
import time
import urllib.request
from sklearn.metrics import confusion_matrix
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Prevent TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print("TensorFlow Version:", tf.__version__)
print("OpenCV Version:", cv2.__version__)

# Download NLTK components
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
    
# Download Haar cascade if not present
cascade_dir = 'haarcascade'
cascade_path = os.path.join(cascade_dir, 'haarcascade_frontalface_default.xml')
if not os.path.exists(cascade_dir):
    os.makedirs(cascade_dir)
if not os.path.exists(cascade_path):
    url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
    urllib.request.urlretrieve(url, cascade_path)
    print("Downloaded Haar Cascade.")
else:
    print("Haar Cascade already exists.")

## Step 1: Explore the FER-2013 Dataset
This block will load your dataset and display class imbalances so you know exactly what is going into the model.

In [ ]:
DATASET_PATH = "dataset/train"

if not os.path.exists(DATASET_PATH):
    print("Error: Please download and extract FER-2013 'train' and 'test' folders into the 'dataset' directory!")
else:
    emotions = os.listdir(DATASET_PATH)
    emotions.sort()
    
    counts = {}
    sample_images = {}
    
    for emotion in emotions:
        emotion_dir = os.path.join(DATASET_PATH, emotion)
        if os.path.isdir(emotion_dir):
            images = os.listdir(emotion_dir)
            counts[emotion] = len(images)
            if len(images) > 0:
                sample_images[emotion] = os.path.join(emotion_dir, images[0])
                
    print(f"Emotion Class Distribution: {counts}")
    
    # Plot Class Imbalance
    plt.figure(figsize=(10, 5))
    plt.bar(counts.keys(), counts.values(), color='skyblue')
    plt.title("FER-2013 Dataset - Class Distribution")
    plt.xlabel("Emotion")
    plt.ylabel("Number of Images")
    plt.show()

    # Show Sample Images
    plt.figure(figsize=(12, 6))
    for i, (emotion, img_path) in enumerate(sample_images.items()):
        plt.subplot(2, 4, i + 1)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        plt.imshow(img, cmap='gray')
        plt.title(f"{emotion.capitalize()}\n{img.shape}")
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()

## Step 2: Build and Train CNN Model
Here we construct our Deep Learning CNN to learn from 48x48 pixel grayscale faces.
*Warning: Running this cell may take some time depending on your CPU/GPU!*

In [ ]:
TRAIN_DIR = 'dataset/train'
TEST_DIR = 'dataset/test'
BATCH_SIZE = 64
    EPOCHS = 50 # Increased for better accuracy with EarlyStopping

def build_model():
    model = Sequential()
    # Block 1
    model.add(Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(48, 48, 1)))
    model.add(BatchNormalization())
    model.add(Conv2D(32, (3, 3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Block 2
    model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Block 3
    model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    # Dense Network
    model.add(Flatten()) # Flatten 2D matrices into a 1D list
    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.5)) 
    model.add(Dense(7, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

if os.path.exists(TRAIN_DIR):
    train_datagen = ImageDataGenerator(
        rescale=1./255, 
        rotation_range=15, 
        width_shift_range=0.15, 
        height_shift_range=0.15,
        horizontal_flip=True,
        zoom_range=0.15,
        shear_range=0.15
    )
    
    val_datagen = ImageDataGenerator(rescale=1./255)

    train_generator = train_datagen.flow_from_directory(
        TRAIN_DIR, target_size=(48, 48), batch_size=BATCH_SIZE, color_mode="grayscale", class_mode='categorical'
    )
    validation_generator = val_datagen.flow_from_directory(
        TEST_DIR, target_size=(48, 48), batch_size=BATCH_SIZE, color_mode="grayscale", class_mode='categorical', shuffle=False
    )

    model = build_model()
    model.summary()

    if not os.path.exists('models'): os.makedirs('models')
    
    callbacks = [
        ModelCheckpoint('models/emotion_model.h5', monitor='val_accuracy', save_best_only=True, mode='max'),
        EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_delta=0.0001),
        tf.keras.callbacks.CSVLogger('models/training_log.csv', append=True)
    ]

    # Feel free to comment this out if the model is already trained
    print("Starting Training...")
    history = model.fit(
        train_generator,
        steps_per_epoch=train_generator.n // train_generator.batch_size,
        epochs=EPOCHS,
        validation_data=validation_generator,
        validation_steps=validation_generator.n // validation_generator.batch_size,
        callbacks=callbacks
    )

else:
    print("Dataset not ready for training!")

### Training Visualization
This cell plots the accuracy and loss curves. You can re-run this anytime without retraining.

In [ ]:
# Run this cell to visualize the training metrics
import pandas as pd
import os
import matplotlib.pyplot as plt

log_path = 'models/training_log.csv'
acc = None
val_acc = None
loss = None
val_loss = None

if 'history' in locals():
    print("Plotting from memory...")
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
elif os.path.exists(log_path):
    print("Plotting from saved CSV log...")
    log_data = pd.read_csv(log_path)
    acc = log_data['accuracy']
    val_acc = log_data['val_accuracy']
    loss = log_data['loss']
    val_loss = log_data['val_loss']
else:
    print("No training history found. You need to train the model first!")

if acc is not None:
    plt.figure(figsize=(12, 4))
    
    # Plot Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(acc, label='Train Accuracy')
    plt.plot(val_acc, label='Val Accuracy')
    plt.legend()
    plt.title('Accuracy')
    
    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(loss, label='Train Loss')
    plt.plot(val_loss, label='Val Loss')
    plt.legend()
    plt.title('Loss')
    
    plt.show()


## Step 3: Define Helper Functions (NLP and Grad-CAM)
Executing this cell will load the functions into memory for our final application.

In [ ]:
# Define NLP Responses
EMOTION_RESPONSES = {
    'angry': ["Take a deep breath. Count to 10.", "It looks like you're frustrated."],
    'disgust': ["Yuck! Something doesn't look right."],
    'fear': ["It's okay to feel scared. You are safe here."],
    'happy': ["You look highly positive today! Keep it up!", "Your smile brightens the room!"],
    'neutral': ["You're completely calm and composed.", "A perfectly balanced emotional state."],
    'sad': ["I'm sorry you are feeling down.", "Don't be sad! Everything will be okay."],
    'surprise': ["Whoa! What just happened? You look shocked!"]
}

def generate_nlp_response(emotion_label, confidence):
    if confidence < 50.0:
        return "I'm not quite sure how you feel right now. My confidence is low.", []
    responses = EMOTION_RESPONSES.get(emotion_label, ["I see a face..."])
    selected_sentence = random.choice(responses)
    tokens = word_tokenize(selected_sentence)
    return selected_sentence, tokens

# Define Grad-CAM functions
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    if tf.reduce_max(heatmap) == 0:
        return heatmap.numpy()
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def overlay_gradcam(img, heatmap, alpha=0.4):
    if len(img.shape) == 2 or img.shape[2] == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    heatmap = np.uint8(255 * heatmap)
    jet = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    jet = cv2.resize(jet, (img.shape[1], img.shape[0]))
    return cv2.addWeighted(img, 1 - alpha, jet, alpha, 0)
    
print("Helper functions defined!")

## Step 4: Run Inference on an Uploaded Image
This cell will provide an upload button for you to select an image.
It will then detect faces, infer emotion, generate an NLP response, and display the result using Matplotlib.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
from PIL import Image, ImageOps

EMOTION_LABELS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
COLOR_MAP = {
    'happy': (0, 255, 0), 'sad': (255, 0, 0), 'angry': (0, 0, 255), 
    'surprise': (0, 255, 255), 'fear': (255, 0, 255), 
    'disgust': (0, 100, 0), 'neutral': (255, 255, 255)
}

model_path = 'models/emotion_model.h5'
if not os.path.exists(model_path):
    print("Cannot find model. Please run the training block fully first!")
else:
    print("Loading AI...")
    loaded_model = tf.keras.models.load_model(model_path)
    
    last_conv_layer_name = None
    for layer in reversed(loaded_model.layers):
        if 'conv2d' in layer.name.lower():
            last_conv_layer_name = layer.name
            break
            
    face_classifier = cv2.CascadeClassifier(cascade_path)

    upload_button = widgets.FileUpload(
        accept='image/*',
        multiple=False
    )
    
    output = widgets.Output()

    def on_upload_change(change):
        with output:
            clear_output()
            if not upload_button.value:
                return
            
            # Extract content safely for ALL different ipywidgets versions
            try:
                if isinstance(upload_button.value, tuple):
                    try:
                        content = upload_button.value[0]['content']
                    except TypeError:
                        content = upload_button.value[0].content
                elif isinstance(upload_button.value, dict):
                    filename = list(upload_button.value.keys())[0]
                    content = upload_button.value[filename]['content']
                else:
                    content = upload_button.value[0].content
            except Exception as e:
                print(f"Error reading uploaded file: {e}")
                return
                
            if isinstance(content, memoryview):
                content = content.tobytes()
            
            # Read image using PIL and convert to OpenCV format
            image = Image.open(io.BytesIO(content)).convert('RGB')
            image = ImageOps.exif_transpose(image)
            frame = np.array(image)
            frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
            
            gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_classifier.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=4, minSize=(30, 30))
            
            if len(faces) == 0:
                print("No faces detected in the image. Try a different image or a clearer face!")
            
            for (x, y, w, h) in faces:
                roi_gray = gray_frame[y:y+h, x:x+w]
                roi_resized = cv2.resize(roi_gray, (48, 48))
                roi_norm = roi_resized.astype('float') / 255.0
                roi_tensor = np.expand_dims(np.expand_dims(roi_norm, axis=-1), axis=0)
                
                preds = loaded_model.predict(roi_tensor, verbose=0)[0]
                label_index = preds.argmax()
                confidence = preds[label_index] * 100
                label_text = EMOTION_LABELS[label_index]
                box_color = COLOR_MAP.get(label_text, (255, 255, 255))
                
                cv2.rectangle(frame, (x, y), (x+w, y+h), box_color, 2)
                cv2.putText(frame, f"{label_text.capitalize()} ({confidence:.1f}%)", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, box_color, 2)
                
                nlp_response, _ = generate_nlp_response(label_text, confidence)
                print(f"Detected Emotion: {label_text.capitalize()} ({confidence:.1f}%)")
                print(f"AI Assistant says: {nlp_response}")
                print("-" * 30)
                
            plt.figure(figsize=(8, 8))
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            plt.imshow(frame_rgb)
            plt.axis('off')
            plt.show()
            
    upload_button.observe(on_upload_change, names='value')
    print("Upload an image to detect emotions!")
    display(upload_button, output)
